# Inspecting the `ourexperimentversionfive/data` DataLoaders

Run the cells from top to bottom. This notebook inspects the **`DataLoader` objects**
produced by `src/ourexperimentversionfive/data` -- split composition, batch shapes,
normalization state, and shuffling behavior. It does not re-derive graph topology or
run known-answer checks on the saved wPLI/edge extraction; see
[`GRAPH_TOPOLOGY_WALKTHROUGH.ipynb`](../GRAPH_TOPOLOGY_WALKTHROUGH.ipynb) in the parent
directory for that. This notebook reads saved data only; it does not regenerate
features or train the GNN.
Current node policy: all eight saved features (five band powers, entropy, mobility, complexity). Each graph has `x.shape == (29, 8)` and alpha-wPLI `edge_attr.shape == (812, 1)`.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import torch_geometric
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/datautils/graphdataversiontwo').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the EEG repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(7)
print('Repository:', ROOT)
print('PyTorch:', torch.__version__, '| PyG:', torch_geometric.__version__)


## 1. Load and validate the raw dataset

`load_dataset` reads the shared `Graph-Liu2024-VersionTwo` saved dataset and runs this
experiment's contract validator (subject/trial counts, node/edge shapes, finiteness,
label balance). Every DataLoader built below reads from this same in-memory object.

In [ ]:
from src.ourexperimentversionfive.data import (
    COMBINATIONS, DEFAULT_COMBINATION, DATASET_DIR, EXPECTED_GRAPHS, EXPECTED_SUBJECT_IDS,
    GraphDataLoaderConfig, get_combination, load_dataset,
)

print('Dataset directory:', DATASET_DIR)
dataset = load_dataset()
metadata = dataset.metadata
print('Samples:', len(dataset), '| expected:', EXPECTED_GRAPHS)
print('Subjects:', metadata['n_samples'] // 40, 'x 40 trials each' )
print('Node variants:', sorted(dataset.nodes.keys()))
print('Edge variants:', sorted(dataset.edges.keys()))
print('Band names:', list(metadata['band_names']))
print('Registered combinations:', sorted(COMBINATIONS), '| default:', DEFAULT_COMBINATION)


## 2. Choose a combination and build one LOSO fold's loaders

`get_combination(name)` returns the same five-function shape (`load_dataset`,
`validate_dataset`, `create_loso_dataloaders`, `create_group_dataloaders`,
`create_within_subject_dataloaders`) regardless of which node/edge/band combination is
selected. `create_loso_dataloaders` builds train/validation/test loaders for one held-out
test subject, with a subject-disjoint validation split drawn from the remaining subjects.

In [ ]:
COMBINATION_NAME = DEFAULT_COMBINATION  # e.g. 'without_csd_alpha_wpli' or 'csd_alpha_wpli'
TEST_SUBJECT_ID = 1                     # any of EXPECTED_SUBJECT_IDS (1..50)
NODE_NORMALIZATION = 'none'             # 'none' or 'zscore'

combination = get_combination(COMBINATION_NAME)
config = GraphDataLoaderConfig(batch_size=8, validation_subjects=5, seed=42,
                                node_normalization=NODE_NORMALIZATION)
bundle = combination.create_loso_dataloaders(
    test_subject_id=TEST_SUBJECT_ID, config=config, dataset=dataset, dataset_validated=True)
print('Combination:', combination.name, '| node:', combination.node_variant,
      '| edge:', combination.edge_variant, '| band:', combination.band_name)
print(bundle.split)


## 3. Split composition and leakage checks

Each fold's train/validation/test subjects must be pairwise disjoint, and every graph
index must land in exactly one split. `create_loso_splits` already asserts this
internally (see `loso_split._validate_split`); we re-check it here directly against the
loaders' own datasets so the DataLoader boundary itself is verified, not just the
`LosoGraphSplit` bookkeeping.

In [ ]:
split = bundle.split
loaders = {'train': bundle.train, 'validation': bundle.validation, 'test': bundle.test}
subject_sets = {
    'train': set(split.train_subject_ids),
    'validation': set(split.validation_subject_ids),
    'test': {split.test_subject_id},
}
assert not (subject_sets['train'] & subject_sets['validation'])
assert not (subject_sets['train'] & subject_sets['test'])
assert not (subject_sets['validation'] & subject_sets['test'])

rows = []
all_labels = np.asarray(dataset.labels)
for name, loader in loaders.items():
    indices = loader.dataset.graph_indices
    labels = all_labels[indices]
    rows.append({
        'split': name,
        'n_subjects': len(subject_sets[name]),
        'n_graphs': len(indices),
        'n_class_0': int((labels == 0).sum()),
        'n_class_1': int((labels == 1).sum()),
    })
display(pd.DataFrame(rows).set_index('split'))
total_graphs = sum(len(l.dataset.graph_indices) for l in loaders.values())
print('Total graphs across splits:', total_graphs, '| expected:', EXPECTED_GRAPHS)
assert total_graphs == EXPECTED_GRAPHS, 'one LOSO fold must partition every graph in the dataset'


## 4. DataLoader configuration and one real batch per split

Only the `train` loader shuffles; `validation`/`test` use a sequential sampler so batches
are reproducible and cover every trial exactly once, in order.

In [ ]:
for name, loader in loaders.items():
    sampler_name = type(loader.sampler).__name__
    print(f'{name:10s} batch_size={loader.batch_size:<3d} sampler={sampler_name}')

print()
batch = next(iter(loaders['train']))
print(batch)
print('num_graphs:', batch.num_graphs)
print('x:', tuple(batch.x.shape), '| edge_index:', tuple(batch.edge_index.shape),
      '| edge_attr:', tuple(batch.edge_attr.shape), '| y:', tuple(batch.y.shape))
print('batch vector (node -> graph):', batch.batch[:5].tolist(), '...')
print('ptr (graph boundaries):', batch.ptr.tolist())
assert batch.x.shape[0] == batch.num_graphs * 29
assert batch.edge_index.shape[1] == batch.num_graphs * 812


## 5. Full pass over the train loader

Iterate every batch once and confirm the loader yields each training graph exactly once
per epoch, with the expected two-class label distribution.

In [ ]:
seen_graphs = 0
label_counts = torch.zeros(2, dtype=torch.long)
n_batches = 0
for batch in loaders['train']:
    seen_graphs += batch.num_graphs
    label_counts += torch.bincount(batch.y, minlength=2)
    n_batches += 1
print(f'{n_batches} batches | {seen_graphs} graphs seen | expected {len(split.train_graph_indices)}')
print('Label counts (class 0, class 1):', label_counts.tolist())
assert seen_graphs == len(split.train_graph_indices)


## 6. Normalization state fitted for this fold

`node_normalization='none'` fits nothing (empty per-subject dictionaries); `'zscore'`
fits one shared mean/std pair from the **training** graphs only, then broadcasts it to
every subject key. Refitting on validation or test indices would leak information across
the LOSO boundary, so `fit_feature_normalization` requires explicit training indices.

In [ ]:
normalization = bundle.normalization
print('Mode:', normalization.mode)
print('Fitted on', len(normalization.fit_graph_indices), 'training graph indices')
if normalization.mode == 'zscore':
    any_subject = next(iter(normalization.mean_by_subject))
    print('Example mean (subject', any_subject, '):', normalization.mean_by_subject[any_subject].numpy())
    print('Example std  (subject', any_subject, '):',
          normalization.standard_deviation_by_subject[any_subject].numpy())
    assert all(torch.equal(m, normalization.mean_by_subject[any_subject])
               for m in normalization.mean_by_subject.values()), 'mean should be identical across subjects'
else:
    assert normalization.mean_by_subject == {} and normalization.fit_graph_indices == ()
    print('Disabled: node features pass through unchanged.')


## 7. Rebuild with z-score normalization and compare one sample

Same fold, same test subject, only `node_normalization` changes. Compare one training
graph's node features before and after.

In [ ]:
zscore_config = GraphDataLoaderConfig(batch_size=8, validation_subjects=5, seed=42,
                                       node_normalization='zscore')
zscore_bundle = combination.create_loso_dataloaders(
    test_subject_id=TEST_SUBJECT_ID, config=zscore_config, dataset=dataset, dataset_validated=True)

raw_graph = bundle.train.dataset[0]
normalized_graph = zscore_bundle.train.dataset[0]
print('Raw x[0]        :', raw_graph.x[0].numpy())
print('Normalized x[0] :', normalized_graph.x[0].numpy())
assert not torch.allclose(raw_graph.x, normalized_graph.x)
assert torch.equal(raw_graph.edge_attr, normalized_graph.edge_attr), 'normalization must not touch edges'
assert torch.equal(raw_graph.y, normalized_graph.y)


## 8. Compare loader shapes across registered combinations

Every entry in `COMBINATIONS` must expose the same `Combination` shape and yield
loaders with the same node/edge tensor shapes for a given test subject -- only the
underlying feature *values* differ (with vs. without CSD).

In [ ]:
rows = []
for name, combo in COMBINATIONS.items():
    combo_dataset = combo.load_dataset() if combo.name != combination.name else dataset
    combo_bundle = combo.create_loso_dataloaders(
        test_subject_id=TEST_SUBJECT_ID,
        config=GraphDataLoaderConfig(batch_size=8, node_normalization='none'),
        dataset=combo_dataset, dataset_validated=True)
    sample = combo_bundle.train.dataset[0]
    rows.append({
        'combination': name, 'node_variant': combo.node_variant,
        'edge_variant': combo.edge_variant, 'band': combo.band_name,
        'x_shape': tuple(sample.x.shape), 'edge_attr_shape': tuple(sample.edge_attr.shape),
    })
display(pd.DataFrame(rows).set_index('combination'))


## 9. Group loaders (no held-out test subject)

`create_group_dataloaders` powers the inner-CV hyperparameter search: an arbitrary pair
of disjoint subject groups, no `test_subject_id` concept. Here we reuse this fold's own
train/validation subject split as an example pairing. `create_within_subject_dataloaders`
follows the same shape for one subject's own trials split by trial index; see
[`GRAPH_TOPOLOGY_WALKTHROUGH.ipynb`](../GRAPH_TOPOLOGY_WALKTHROUGH.ipynb) section 10 for
a full worked example of that variant.

In [ ]:
group_train_loader, group_validation_loader, group_normalization = combination.create_group_dataloaders(
    dataset=dataset,
    train_subject_ids=split.train_subject_ids,
    validation_subject_ids=split.validation_subject_ids,
    config=GraphDataLoaderConfig(batch_size=8, node_normalization='none'),
    dataset_validated=True,
)
print('Group train graphs:', len(group_train_loader.dataset.graph_indices))
print('Group validation graphs:', len(group_validation_loader.dataset.graph_indices))
assert not (set(group_train_loader.dataset.graph_indices)
            & set(group_validation_loader.dataset.graph_indices))


## 10. What this notebook confirms

| Check | Where |
|---|---|
| Raw dataset loads and passes the experiment's contract validator | Section 1 |
| LOSO loaders build for an arbitrary combination/test subject | Section 2 |
| Train/validation/test subjects and graph indices are disjoint and complete | Section 3 |
| Train loader shuffles, validation/test do not; batch tensor shapes match the PyG contract | Section 4 |
| A full epoch over train yields every training graph exactly once | Section 5 |
| Normalization fits only on training indices and broadcasts identically across subjects | Section 6 |
| z-score normalization changes node features but never edges or labels | Section 7 |
| Every registered combination exposes the same loader shape contract | Section 8 |
| Group (inner-CV-style) loaders build from arbitrary disjoint subject sets | Section 9 |

What remains outside this notebook's scope: whether the saved wPLI/node-feature *values*
are themselves scientifically correct (see `GRAPH_TOPOLOGY_WALKTHROUGH.ipynb`), and
whether training on these loaders actually produces a good model (see
`training/README.md`).